In [ ]:

import torch
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

# Load pretrained ResNet
model = models.resnet50(pretrained=True)
model.eval()

# Pick last conv layer
target_layer = model.layer4[2].conv3

# Simple image
url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
img = Image.open(requests.get(url, stream=True).raw)

# Preprocess
transform = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = transform(img).unsqueeze(0)

# Forward pass
output = model(input_tensor)
pred_class = output.argmax().item()

# Register hooks
activations, gradients = [], []
def forward_hook(module, inp, out): activations.append(out)
def backward_hook(module, grad_in, grad_out): gradients.append(grad_out[0])
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

# Backward
score = output[0, pred_class]
model.zero_grad()
score.backward()

# Grad-CAM
weights = gradients[0].mean(dim=(2,3), keepdim=True)
cam = (weights * activations[0]).sum(dim=1).relu()
cam = torch.nn.functional.interpolate(cam.unsqueeze(0), size=(224,224), mode="bilinear")[0,0]

# Normalize
cam = (cam - cam.min()) / (cam.max() - cam.min())

# Show
plt.imshow(img)
plt.imshow(cam.detach().numpy(), cmap='jet', alpha=0.5)
plt.title("Grad-CAM Heatmap")
plt.axis("off")
plt.show()
